<a href="https://colab.research.google.com/github/Dktuthan1472017/AIL303m_SE204293_DuongDuyLoi/blob/main/SE204203_DuongDuyLoi_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cơ sở toán học

Thuật toán Gaussian Naive Bayes dựa trên công thức Bayes:

$$P(y \mid X) = \frac{P(y) \cdot P(X \mid y)}{P(X)}$$

Giả định các đặc trưng $x_1, x_2, \dots, x_n$ độc lập lập điều kiện khi biết $y$, xác suất khả năng (likelihood) được tính bằng phân bố chuẩn (Gaussian):

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi \sigma_y^2}} \exp\left( -\frac{(x_i - \mu_y)^2}{2\sigma_y^2} \right)$$

Để tránh lỗi tràn số dưới (underflow) khi nhân nhiều xác suất nhỏ, chúng ta chuyển sang tính Log-Likelihood:

$$\log P(y \mid X) \propto \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)$$

In [ ]:
!pip install -q streamlit scikit-learn pandas matplotlib
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) over (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

In [106]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

# ==============================================================================
# 1. CẤU HÌNH TRANG & TIÊU ĐỀ
# ==============================================================================
st.set_page_config(
    page_title="SE204203_DuongDuyLoi_Streamlit",
    page_icon="🌸",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("SE204203_DuongDuyLoi_Streamlit")
st.caption("Machine Learning Model: **Gaussian Naive Bayes (sklearn)** | Framework: **Streamlit**")
st.markdown(
    """
> **Hướng dẫn:** Sử dụng thanh kéo ở cột bên trái (bước nhảy 0.01). Bạn có thể **double-click** vào giá trị trên thanh kéo để nhập số trực tiếp từ bàn phím, sau đó nhấn nút **"Dự đoán ngay"** để hiển thị kết quả.
"""
)
st.markdown("---")

# ==============================================================================
# 2. TẢI DỮ LIỆU & HUẤN LUYỆN MÔ HÌNH (CACHED)
# ==============================================================================
@st.cache_resource
def load_data_and_train_model():
    """Tải tập dữ liệu Iris và huấn luyện mô hình GaussianNB một lần duy nhất."""
    iris = load_iris()
    X = iris.data
    y = iris.target
    feature_names = iris.feature_names
    target_names = iris.target_names

    # Huấn luyện mô hình chuẩn GaussianNB của sklearn
    model = GaussianNB()
    model.fit(X, y)

    # Chia tập dữ liệu 80/20 để đánh giá độ chính xác
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    eval_model = GaussianNB()
    eval_model.fit(X_train, y_train)
    test_accuracy = eval_model.score(X_test, y_test)

    # Tạo DataFrame để trực quan hóa dữ liệu
    df = pd.DataFrame(X, columns=feature_names)
    df["species"] = [target_names[i] for i in y]

    return model, iris, test_accuracy, df

model, iris_dataset, test_acc, df_iris = load_data_and_train_model()

# ==============================================================================
# 3. THANH SIDEBAR (Rút gọn chỉ chứa thông tin mô hình cơ bản)
# ==============================================================================
with st.sidebar:
    st.header("ℹ️ Model Information")
    st.markdown("- **Project:** SE204203_DuongDuyLoi_Streamlit")
    st.markdown("- **Algorithm:** Gaussian Naive Bayes")
    st.metric(label="Test Accuracy (80/20 Split)", value=f"{test_acc * 100:.2f}%")
    st.markdown("---")
    with st.expander("📊 View Iris Raw Data"):
        st.dataframe(df_iris)

IMAGE_URLS = {
    "setosa": "https://upload.wikimedia.org/wikipedia/commons/a/a7/Irissetosa1.jpg",
    "versicolor": "https://upload.wikimedia.org/wikipedia/commons/2/27/Blue_Flag%2C_Ottawa.jpg",
    "virginica": "https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg"
}

# ==============================================================================
# 4. GIAO DIỆN CHÍNH CHIA 2 CỘT
# ==============================================================================
col_left, col_right = st.columns([1, 1], gap="large")

# Khởi tạo hoặc lấy trạng thái dự đoán để hiển thị tỉ lệ động ở cột trái
if "predicted" not in st.session_state:
    st.session_state.predicted = False
    st.session_state.probabilities = np.array([0.0, 0.0, 0.0])
    st.session_state.predicted_species = ""

# --- CỘT BÊN TRÁI: NHẬP THÔNG TIN MẪU HOA & TỈ LỆ DỰ ĐOÁN ---
with col_left:
    st.subheader("📥 Nhập thông tin mẫu hoa")

    sepal_length = st.slider(
        label="Chiều dài lá đài (Sepal Length - cm)",
        min_value=4.00, max_value=8.00, value=5.10, step=0.01, format="%.2f"
    )
    sepal_width = st.slider(
        label="Chiều rộng lá đài (Sepal Width - cm)",
        min_value=2.00, max_value=4.50, value=3.50, step=0.01, format="%.2f"
    )
    petal_length = st.slider(
        label="Chiều dài cánh hoa (Petal Length - cm)",
        min_value=1.00, max_value=7.00, value=1.40, step=0.01, format="%.2f"
    )
    petal_width = st.slider(
        label="Chiều rộng cánh hoa (Petal Width - cm)",
        min_value=0.10, max_value=2.50, value=0.20, step=0.01, format="%.2f"
    )

    st.markdown("")
    predict_btn = st.button("🔍 Dự đoán ngay", type="primary", use_container_width=True)
    st.markdown("---")

    # THỰC HIỆN TÍNH TOÁN DỰ ĐOÁN ĐỂ LẤY TỈ LỆ TRƯỚC
    input_data = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    probabilities = model.predict_proba(input_data)[0]
    prediction = model.predict(input_data)[0]
    predicted_species = iris_dataset.target_names[prediction]

    if predict_btn:
        st.session_state.predicted = True
        st.session_state.probabilities = probabilities
        st.session_state.predicted_species = predicted_species

    # HIỂN THỊ TỈ LỆ NGAY BÊN DƯỚI CHỖ KÉO THẢ CHỌN SỐ LIỆU
    st.subheader("📊 Tỉ lệ dự đoán")
    if st.session_state.predicted:
        prob_dict = {
            iris_dataset.target_names[i]: float(st.session_state.probabilities[i])
            for i in range(len(iris_dataset.target_names))
        }

        chart_data = pd.DataFrame(
            {"Loài hoa": list(prob_dict.keys()), "Xác suất": list(prob_dict.values())}
        ).set_index("Loài hoa")
        st.bar_chart(chart_data)

        for species_name, prob in prob_dict.items():
            st.write(f"- **{species_name.capitalize()}**: `{prob * 100:.2f}%`")
            st.progress(float(prob))
    else:
        st.info("💡 Nhấn nút **'Dự đoán ngay'** ở trên để cập nhật tỉ lệ chi tiết tại đây.")

# --- CỘT BÊN PHẢI: KẾT QUẢ DỰ ĐOÁN & BIỂU ĐỒ VỊ TRÍ CHỌN ---
with col_right:
    st.subheader("📤 Kết quả dự đoán")

    if st.session_state.predicted:
        current_pred_species = st.session_state.predicted_species
        # 1. Hiển thị loài hoa dự đoán được
        st.success(f"### Loài hoa dự đoán: **{current_pred_species.upper()}**")

        # 2. Nạp ảnh minh họa của loài hoa tương ứng từ Wikipedia
        try:
            img_url = IMAGE_URLS.get(current_pred_species.lower())
            if img_url:
                st.image(img_url, caption=f"Hình ảnh minh họa: Iris {current_pred_species.capitalize()}", width=350)
            else:
                st.warning("⚠️ Không tìm thấy ảnh tương ứng của loài hoa này.")
        except Exception as e:
            st.warning(f"⚠️ Không thể hiển thị ảnh của loài hoa: {e}")
    else:
        st.info("💡 Kết quả phân loại loài hoa sẽ hiển thị tại đây sau khi nhấn dự đoán.")

    st.markdown("---")
    # 3. BIỂU ĐỒ THỂ HIỆN VỊ TRÍ ĐANG CHỌN (CURRENT) - HIỆN NGAY BÊN DƯỚI HÌNH HOA
    st.subheader("📈 Vị trí điểm chọn (Feature Space Plot)")
    fig, ax = plt.subplots(figsize=(6, 4.2))
    for sp in df_iris["species"].unique():
        subset = df_iris[df_iris["species"] == sp]
        ax.scatter(subset["petal length (cm)"], subset["petal width (cm)"], label=sp.capitalize(), alpha=0.6)

    # Hiển thị điểm X đỏ trực quan thể hiện đúng vị trí đang chọn
    ax.scatter(petal_length, petal_width, color="red", marker="X", s=180, label="Current Input")
    ax.set_xlabel("Petal Length (cm)")
    ax.set_ylabel("Petal Width (cm)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    st.pyplot(fig)

Overwriting app.py


In [107]:
import subprocess
import time
import re

# 1.Streamlit
cmd_streamlit = [
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
]
subprocess.Popen(cmd_streamlit)
time.sleep(2)

# 2. Cloudflare Tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting tunnel, please wait...")
found = False
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("\n" + "="*50)
        print(f"👉 CLICK TO OPEN APP: {url}")
        print("="*50 + "\n")
        found = True
        break

if not found:
    print("Could not retrieve link automatically. Please check terminal output.")

Starting tunnel, please wait...

👉 CLICK TO OPEN APP: https://thumbnail-roulette-award-wellington.trycloudflare.com

